# E2: BoW + Linear SVM

**Câu hỏi:** Giữ nguyên biểu diễn BoW như E1, đổi Naive Bayes thành Linear SVM thì Toxic F1 thay đổi thế nào?

**Đầu vào:** `message_segmented`, tạo từ `message_clean` bằng VnCoreNLP. Không dùng `message_normalized` và các cột kiểm tra nhãn.

**Chia dữ liệu theo kế hoạch NLP_N12:** 70/15/15 theo `post_id`, seed 42, cố gắng giữ phân bố nhãn. Không bài đăng nào nằm ở hai tập.

**Chọn cấu hình:** chỉ dùng Toxic F1 trên Validation; Test chỉ chạy một lần ở cuối.

**Lưu ý khi đọc kết quả:**
- Validation và Test có khoảng 21,7% toxic, cao hơn phân bố tự nhiên, nên điểm số chưa đại diện cho dữ liệu thực tế.
- Ngưỡng được chọn trên chính Validation nên điểm Validation hơi lạc quan.

## 1. Cài đặt

Cần Java (đã có sẵn trên máy này) để chạy VnCoreNLP.

In [ ]:
%pip install -q scikit-learn pandas openpyxl matplotlib joblib py_vncorenlp

## 2. Cấu hình

In [ ]:
import json
import os
import re
import unicodedata
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

EXPERIMENT = "E2"
SEED = 42
RUN_TEST = False  # Chỉ bật một lần ở cuối, sau khi đã chốt mọi cấu hình bằng Validation

BASE = Path.cwd().resolve()
DATA = BASE / "DATA_NLP_N12_train.xlsx"
SPLIT_DIR = BASE / "data_splits"   # E1-E10 dùng chung bản tách từ và danh sách chia tập
VNCORE_DIR = BASE / "vncorenlp"
OUT = BASE / "results" / EXPERIMENT

assert DATA.exists(), f"Không thấy {DATA}. Hãy mở notebook từ thư mục BTL1."
for folder in [SPLIT_DIR, VNCORE_DIR, OUT]:
    folder.mkdir(parents=True, exist_ok=True)

## 3. Đọc dữ liệu

In [ ]:
df = pd.read_excel(
    DATA, sheet_name="TRAIN_DATA",
    dtype={"sample_id": str, "post_id": str, "comment_id": str},
)
df["message_clean"] = df["message_clean"].fillna("")  # dòng chỉ có emoji đã bị xóa thành rỗng

assert df["sample_id"].is_unique
assert not df.duplicated(["post_id", "comment_id"]).any()
assert set(df["label"]) == {0, 1}
assert not df["post_id"].str.startswith("EXT_").any(), "Không dùng dữ liệu ngoài trong file train"

print(f"{len(df):,} bình luận | {df['post_id'].nunique():,} bài đăng | "
      f"toxic {df['label'].sum():,} ({df['label'].mean():.2%}) | rỗng {(df['message_clean'] == '').sum()}")

## 4. Tách từ bằng VnCoreNLP

Lần đầu tải 3 file và tách từ gần 10.000 dòng; kết quả được lưu lại cho lần sau.

In [ ]:
SEG_FILE = SPLIT_DIR / "message_segmented.csv"


def load_segmenter():
    """py_vncorenlp.download_model cần wget nên lỗi trên Windows; chỉ tải đúng 3 file mà wseg cần."""
    import py_vncorenlp

    root = "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/"
    for name in ["VnCoreNLP-1.2.jar", "models/wordsegmenter/vi-vocab", "models/wordsegmenter/wordsegmenter.rdr"]:
        path = VNCORE_DIR / name
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            print("Đang tải", name)
            urllib.request.urlretrieve(root + name, path)
    # VnCoreNLP đổi thư mục làm việc khi khởi tạo, phải trả lại để các đường dẫn sau không sai
    model = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORE_DIR))
    os.chdir(BASE)
    return model


def segment_text(segmenter, text):
    """VnCoreNLP làm hỏng ký tự ngoài BMP (🤣 thành 藍), nên chỉ tách phần chữ và giữ nguyên emoji."""
    pieces = []
    for part in re.split(r"([𐀀-􏿿]+)", text):
        if re.fullmatch(r"[𐀀-􏿿]+", part):
            pieces.append(part)
        elif part.strip():
            pieces.extend(segmenter.word_segment(part))
    return " ".join(pieces)


def cached_segments():
    if not SEG_FILE.exists():
        return None
    cached = pd.read_csv(SEG_FILE, dtype=str, keep_default_na=False).set_index("sample_id")
    if set(cached.index) != set(df["sample_id"]):
        return None
    # Văn bản đổi thì bản tách từ cũ không còn đúng
    same = cached.loc[df["sample_id"], "message_clean"].values == df["message_clean"].values
    return cached.loc[df["sample_id"], "message_segmented"].values if same.all() else None


segments = cached_segments()
if segments is None:
    segmenter = load_segmenter()
    # Tách từ từng dòng, không chạm vào message_clean; chạy một lần rồi lưu lại cho các notebook sau
    segments = [segment_text(segmenter, t) for t in df["message_clean"]]
    pd.DataFrame({
        "sample_id": df["sample_id"], "message_clean": df["message_clean"], "message_segmented": segments,
    }).to_csv(SEG_FILE, index=False, encoding="utf-8")

df["message_segmented"] = segments

# Tự kiểm tra: tách từ chỉ được thêm khoảng trắng và dấu nối "_", không được mất hay đổi ký tự
# (riêng vị trí dấu thanh được VnCoreNLP chuẩn hóa, ví dụ hóa thành hoá, nên so sánh sau khi bỏ dấu)
def bare(col):
    return col.map(lambda t: re.sub(r"[\s_]", "", "".join(
        c for c in unicodedata.normalize("NFD", t) if not unicodedata.combining(c))))


broken = df[bare(df["message_clean"]) != bare(df["message_segmented"])]
print("Dòng bị đổi ký tự khi tách từ:", len(broken))
assert len(broken) == 0, broken[["message_clean", "message_segmented"]].head()
df[["message_clean", "message_segmented"]].sample(5, random_state=SEED)

## 5. Chia Train/Validation/Test

Lần đầu sinh `data_splits/split_ids.csv`; từ lần sau mọi notebook đều đọc lại đúng danh sách này.

In [ ]:
SPLIT_FILE = SPLIT_DIR / "split_ids.csv"

if SPLIT_FILE.exists():
    ids = pd.read_csv(SPLIT_FILE, dtype=str)
    assert set(ids["sample_id"]) == set(df["sample_id"]), "split_ids.csv không khớp dữ liệu hiện tại"
    df["split"] = df["sample_id"].map(ids.set_index("sample_id")["split"])
else:
    # 20 phần theo post_id: 14 phần Train, 3 phần Validation, 3 phần Test = 70/15/15
    fold = np.zeros(len(df), dtype=int)
    splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
    for i, (_, part) in enumerate(splitter.split(df, df["label"], groups=df["post_id"])):
        fold[part] = i
    df["split"] = np.select([fold < 14, fold < 17], ["train", "val"], default="test")
    df[["sample_id", "post_id", "split"]].to_csv(SPLIT_FILE, index=False, encoding="utf-8")

# Tự kiểm tra: không bài đăng nào nằm ở hai tập, tỷ lệ đúng 70/15/15 và phân bố nhãn gần nhau
assert (df.groupby("post_id")["split"].nunique() == 1).all()
assert np.allclose(df["split"].value_counts(normalize=True)[["train", "val", "test"]], [0.70, 0.15, 0.15], atol=0.02)
assert df.groupby("split")["label"].mean().sub(df["label"].mean()).abs().max() < 0.02

summary = df.groupby("split").agg(
    binh_luan=("label", "size"), bai_dang=("post_id", "nunique"), toxic=("label", "sum"), ty_le_toxic=("label", "mean"),
).loc[["train", "val", "test"]]

text = df["message_segmented"].str.lower()
train_texts = set(text[df["split"] == "train"]) - {""}
for name in ["val", "test"]:
    print(f"Trùng văn bản với Train ở {name}:", text[df["split"] == name].isin(train_texts).sum())

train, val, test = (df[df["split"] == s] for s in ["train", "val", "test"])
summary.round(4)

## 6. Hàm dùng chung

In [ ]:
# Mặc định CountVectorizer bỏ từ 1 ký tự và emoji, làm mất các tín hiệu như "đ", "m", 🖕
TOKEN_PATTERN = r"\w+|[^\w\s]"

tokens = CountVectorizer(token_pattern=TOKEN_PATTERN).build_analyzer()("Mày ngu_vl đ 🖕")
assert tokens == ["mày", "ngu_vl", "đ", "🖕"], tokens


def best_threshold(y, scores):
    """Ngưỡng cho Toxic F1 cao nhất; chỉ được gọi trên Validation."""
    precision, recall, thresholds = precision_recall_curve(y, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    i = int(np.argmax(f1))
    return float(thresholds[i]), float(f1[i])


def evaluate(y, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "toxic_precision": precision_score(y, pred, zero_division=0),
        "toxic_recall": recall_score(y, pred, zero_division=0),
        "toxic_f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y, pred),
        "pr_auc": average_precision_score(y, scores),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def plot_results(y, scores, threshold, title):
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay.from_predictions(
        y, (scores >= threshold).astype(int), labels=[0, 1],
        display_labels=["Non-toxic", "Toxic"], cmap="Blues", colorbar=False, ax=left,
    )
    left.set_title(f"Ma trận nhầm lẫn: {title}")
    PrecisionRecallDisplay.from_predictions(y, scores, ax=right, plot_chance_level=True, name=EXPERIMENT)
    right.set_title(f"Đường Precision-Recall lớp Toxic: {title}")
    fig.tight_layout()
    fig.savefig(OUT / f"{title.lower()}_charts.png", dpi=150)
    plt.show()

## 7. Mô hình và lưới tham số

In [ ]:
MODEL_NAME = "BoW + Linear SVM"


def make_model():
    return Pipeline([
        ("vec", CountVectorizer(token_pattern=TOKEN_PATTERN)),
        ("clf", LinearSVC(dual="auto", max_iter=10000, random_state=SEED)),
    ])


def toxic_scores(model, texts):
    """Khoảng cách có dấu tới siêu phẳng: 0 là ranh giới mặc định của SVM."""
    return model.decision_function(texts)


GRID = {
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df": [1, 2, 3],
    "vec__max_features": [None, 20000],
    "clf__C": [0.001, 0.01, 0.1, 1.0, 10.0],
    "clf__class_weight": [None, "balanced"],  # balanced = phạt nặng hơn khi bỏ sót lớp Toxic
}
print("Số cấu hình:", len(ParameterGrid(GRID)))

## 8. Dò tham số trên Validation

In [ ]:
rows = []
for params in ParameterGrid(GRID):
    model = make_model().set_params(**params).fit(train["message_segmented"], train["label"])
    scores = toxic_scores(model, val["message_segmented"])
    threshold, tuned_f1 = best_threshold(val["label"], scores)
    rows.append({
        **params, "params": params, "val_f1": tuned_f1, "threshold": threshold,
        "val_f1_threshold_0": f1_score(val["label"], (scores >= 0).astype(int), zero_division=0),
        "val_pr_auc": average_precision_score(val["label"], scores),
    })

grid = pd.DataFrame(rows).sort_values(["val_f1", "val_pr_auc"], ascending=False).reset_index(drop=True)
grid.drop(columns="params").to_csv(OUT / "grid_search.csv", index=False, encoding="utf-8-sig")
grid.drop(columns="params").head(10)

## 9. Huấn luyện lại với cấu hình tốt nhất

In [ ]:
best = grid.loc[0, "params"]
THRESHOLD = float(grid.loc[0, "threshold"])
print("Cấu hình tốt nhất:", best, "| ngưỡng:", round(THRESHOLD, 4))

model = make_model().set_params(**best).fit(train["message_segmented"], train["label"])
val_scores = toxic_scores(model, val["message_segmented"])

# Mô hình tham chiếu của kế hoạch: luôn đoán lớp phổ biến nhất, không tính vào E1-E10
majority = DummyClassifier(strategy="most_frequent").fit(train["message_segmented"], train["label"])
majority_pred = majority.predict(val["message_segmented"])

metrics = {
    "Train": evaluate(train["label"], toxic_scores(model, train["message_segmented"]), THRESHOLD),
    "Validation, ngưỡng 0": evaluate(val["label"], val_scores, 0.0),
    "Validation, ngưỡng đã chọn": evaluate(val["label"], val_scores, THRESHOLD),
    "Lớp phổ biến nhất": evaluate(val["label"], majority_pred.astype(float), 0.5),
}
pd.DataFrame(metrics).T.round(4)

In [ ]:
plot_results(val["label"], val_scores, THRESHOLD, "Validation")

## 10. Từ ảnh hưởng nhất

In [ ]:
# Hệ số SVM: dương là đẩy về Toxic
vocab = model.named_steps["vec"].get_feature_names_out()
weights = pd.Series(model.named_steps["clf"].coef_[0], index=vocab).sort_values()
pd.DataFrame({
    "nghieng_toxic": weights.index[::-1][:20], "trong_so_toxic": weights.values[::-1][:20].round(3),
    "nghieng_non_toxic": weights.index[:20], "trong_so_non_toxic": weights.values[:20].round(3),
})

## 11. Phân tích lỗi trên Validation

In [ ]:
val_errors = val.assign(score=val_scores, pred=(val_scores >= THRESHOLD).astype(int))
val_errors = val_errors[val_errors["pred"] != val_errors["label"]]
val_errors["loai_loi"] = np.where(val_errors["label"] == 1, "FN: bỏ sót toxic", "FP: báo nhầm toxic")
val_errors = val_errors.sort_values("score", key=np.abs, ascending=False)[
    ["sample_id", "post_id", "loai_loi", "score", "post_message", "parent_message", "message_clean"]
]
val_errors.to_csv(OUT / "val_errors.csv", index=False, encoding="utf-8-sig")
print(val_errors["loai_loi"].value_counts().to_dict())
val_errors[["loai_loi", "score", "message_clean"]].head(15)

## 12. So sánh với E1

In [ ]:
# RQ1: so E2 với E1 trên cùng Validation. Cần chạy notebook E1 trước.
e1_file = BASE / "results" / "E1" / "metrics.json"
if e1_file.exists():
    e1 = json.loads(e1_file.read_text(encoding="utf-8"))["metrics"]
    keys = ["toxic_precision", "toxic_recall", "toxic_f1", "macro_f1", "accuracy", "pr_auc"]
    name = "Validation, ngưỡng đã chọn"
    display(pd.DataFrame({"E1": {k: e1[name][k] for k in keys}, "E2": {k: metrics[name][k] for k in keys}}).T.round(4))
else:
    print("Chưa có results/E1/metrics.json")

## Lưu kết quả

In [ ]:
joblib.dump({"model": model, "threshold": THRESHOLD}, OUT / "model.joblib")
report = {
    "experiment": EXPERIMENT, "model": MODEL_NAME, "input": "message_segmented", "seed": SEED,
    "best_params": best,
    "threshold": THRESHOLD, "metrics": metrics,
}
(OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu vào", OUT)

## Đánh giá cuối trên Test

Chỉ chạy khi đã chốt cấu hình. Không quay lại chỉnh tham số dựa trên kết quả này.

In [ ]:
if RUN_TEST:
    test_scores = toxic_scores(model, test["message_segmented"])
    report["metrics"]["Test"] = evaluate(test["label"], test_scores, THRESHOLD)
    (OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    test.assign(score=test_scores, pred=(test_scores >= THRESHOLD).astype(int))[
        ["sample_id", "post_id", "label", "pred", "score", "message_clean"]
    ].to_csv(OUT / "test_predictions.csv", index=False, encoding="utf-8-sig")
    plot_results(test["label"], test_scores, THRESHOLD, "Test")
    display(pd.DataFrame({"Test": report["metrics"]["Test"]}).T.round(4))
else:
    print("Chưa đánh giá Test. Đặt RUN_TEST = True sau khi đã chốt cấu hình, rồi chạy lại toàn bộ notebook.")